In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

raw_root = "s3://l1l2bucket/data/raw/phantom"

catalog = "crypto_lakehouse"
bronze_schema = "bronze"

display(dbutils.fs.ls(raw_root))

path,name,size,modificationTime
s3://l1l2bucket/data/raw/phantom/ingestion_manifest/,ingestion_manifest/,0,1791559811474
s3://l1l2bucket/data/raw/phantom/sol_balance/,sol_balance/,0,1791559811474
s3://l1l2bucket/data/raw/phantom/sol_transactions/,sol_transactions/,0,1791559811474


####PHANTOM SOL BALANCE

In [0]:
sol_balance_schema = StructType([
    StructField("asset", StringType(), True),
    StructField("balance", DecimalType(38, 18), True),
    StructField("network", StringType(), True),
    StructField("wallet_address", StringType(), True)
])

sol_balance_raw_df = (
    spark.read
    .schema(sol_balance_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/sol_balance/")
)

sol_balance_bronze_df = (
    sol_balance_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("wallet_name",lit("Phantom"))
    .withColumn("source_type",lit("Phantom Balance"))
    .withColumn("ingested_at",current_timestamp())
    .dropDuplicates([
        "wallet_address",
        "asset",
        "_source_file"
    ])
)

print("Phantom SOL balance rows:", sol_balance_bronze_df.count())
sol_balance_bronze_df.printSchema()
display(sol_balance_bronze_df)

Phantom SOL balance rows: 1
root
 |-- asset: string (nullable = true)
 |-- balance: decimal(38,18) (nullable = true)
 |-- network: string (nullable = true)
 |-- wallet_address: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- wallet_name: string (nullable = false)
 |-- source_type: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



asset,balance,network,wallet_address,source_file_name,_source_file,wallet_name,source_type,ingested_at
SOL,0.025388000000000000,SOLANA,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,sol_balance_20261008T143029_084070Z_6f31d726.json,s3://l1l2bucket/data/raw/phantom/sol_balance/sol_balance_20261008T143029_084070Z_6f31d726.json,Phantom,Phantom Balance,2026-10-09T15:30:18.211Z


In [0]:
(
    sol_balance_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.phantom_sol_balance"
    )
)

####Phantom SOL Transactions

In [0]:
sol_transactions_raw_df = (
    spark.read
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/sol_transactions/")
)

print(
    "Phantom SOL transaction rows:",
    sol_transactions_raw_df.count()
)

sol_transactions_raw_df.printSchema()

display(
    sol_transactions_raw_df.limit(5)
)

Phantom SOL transaction rows: 7
root
 |-- signature: string (nullable = true)
 |-- transaction: struct (nullable = true)
 |    |-- blockTime: long (nullable = true)
 |    |-- meta: struct (nullable = true)
 |    |    |-- computeUnitsConsumed: long (nullable = true)
 |    |    |-- costUnits: long (nullable = true)
 |    |    |-- err: struct (nullable = true)
 |    |    |    |-- InsufficientFundsForRent: struct (nullable = true)
 |    |    |    |    |-- account_index: long (nullable = true)
 |    |    |-- fee: long (nullable = true)
 |    |    |-- innerInstructions: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- logMessages: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- postBalances: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- postTokenBalances: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |

signature transaction 5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm List(1782684012, List(750, 2688, null, 11000, List(), List(Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success), List(1102058845764807, 20103757, 25388000, 335789966, 42706560, 1, 1), List(), List(1102059196963807, 20103757, 0, 9989966, 42706560, 1, 1), List(), List(), List(null, null)), 429536564, List(List(List(List(5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9, true, transaction, true), List(HFWY9uK3BMuLhaxUQfsFFGX7XGm7LqC8GLNSEt39CLMe, false, transaction, true), List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, false, transaction, true), List(36Unv5vbfhvxKgX1mq7u6XPcwnCQh7NXyVAg2xCMfUXT, false, transaction, true), List(SysvarRecentB1ockHashes11111111111111111111, false, transaction, false), List(11111111111111111111111111111111, false, transaction, false), List(ComputeBudget111111111111111111111111111111, false, transaction, false)), List(List(null, null, List(List(null, null, HFWY9uK3BMuLhaxUQfsFFGX7XGm7LqC8GLNSEt39CLMe, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9, SysvarRecentB1ockHashes11111111111111111111, null), advanceNonce), system, 11111111111111111111111111111111, 1), List(List(), 3QAwFKa3MJAs, null, null, ComputeBudget111111111111111111111111111111, 1), List(List(), FKuNA3, null, null, ComputeBudget111111111111111111111111111111, 1), List(null, null, List(List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, 25388000, null, null, null, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9), transfer), system, 11111111111111111111111111111111, 1), List(null, null, List(List(36Unv5vbfhvxKgX1mq7u6XPcwnCQh7NXyVAg2xCMfUXT, 325800000, null, null, null, 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9), transfer), system, 11111111111111111111111111111111, 1)), D1g2vyB99L3wVqCN13FejqaZV9U9n35Md6AEWYYiUmYa), List(5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm)), 2144, legacy) 2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M List(1773409311, List(450, 1784, null, 80000, List(), List(Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program ComputeBudget111111111111111111111111111111 invoke [1], Program ComputeBudget111111111111111111111111111111 success, Program 11111111111111111111111111111111 invoke [1], Program 11111111111111111111111111111111 success), List(0, 106414822, 1, 1), List(), List(6494922, 99999900, 1, 1), List(), List(), List(null, null)), 406146780, List(List(List(List(FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7, true, transaction, true), List(7q3KWGdYaVsHEUZvDoZcJ9zCFNV3UYDY7ibLJq3aW8Ub, false, transaction, true), List(11111111111111111111111111111111, false, transaction, false), List(ComputeBudget111111111111111111111111111111, false, transaction, false)), List(List(List(), 3b1H8Rq1T3d1, null, null, ComputeBudget111111111111111111111111111111, 1), List(List(), LKoyXd, null, null, ComputeBudget111111111111111111111111111111, 1), List(null, null, List(List(7q3KWGdYaVsHEUZvDoZcJ9zCFNV3UYDY7ibLJq3aW8Ub, 6414922, null, null, null, FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7), transfer), system, 11111111111111111111111111111111, 1)), mX5nfF6zXMDEowJJB3uLdMahQXcLHWKWQHsVk27xHGm), List(2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M)), 254, legacy) 3xjSrZJi8e4cyRP97uEZFPM1gQKmWXdPBNfigFFQUjRarYk5hFFd9CsCo8HDKcCzYFQYrC8iJDo4Yhp

In [0]:
# ========================================================
# PHANTOM SOL TRANSACTIONS
# ========================================================

sol_transactions_bronze_df = (
    sol_transactions_raw_df
    .select(
        "*",

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "wallet_address",
        lit(
            "FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7"
        )
    )
    .withColumn(
        "wallet_name",
        lit("Phantom")
    )
    .withColumn(
        "network",
        lit("SOLANA")
    )
    .withColumn(
        "source_type",
        lit("Phantom Transaction")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "signature"
    ])
)

print(
    "Phantom SOL transaction rows:",
    sol_transactions_bronze_df.count()
)

display(
    sol_transactions_bronze_df.select(
        "signature",
        col("transaction.blockTime")
        .alias("block_time"),

        col("transaction.meta.fee")
        .alias("fee_lamports"),

        col("transaction.meta.err")
        .alias("transaction_error"),

        "wallet_address",
        "_source_file"
    )
    .orderBy(
        col("block_time").desc()
    )
)

Phantom SOL transaction rows: 7


signature,block_time,fee_lamports,transaction_error,wallet_address,_source_file
5EBywo7VUSKhMDUZW714Mmdfqb3dH4HaRXjf6FoY3vKZQw1RKeirGBUtUjES8QR4xMDep5br7c6qegrw58WkHKBm,1782684012,11000,null,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json
2qLcZAvFqTytyA9cYBBreQRM9AzJoPjqrkSKY88rJhb2UYKHmSzM4LttjjNz5PZUo1AN5tSekaTf1wFQ9U9u8e4M,1773409311,80000,null,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json
4B1MJE5DHiKKpzmezWYDaJG7EFrPU8WgW5GTqrNZXYxxKDQKk5XbjY5gzUTk6of364xLBYRUgwjyPCXsqVQxtA9m,1773408984,5000,null,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json
3xjSrZJi8e4cyRP97uEZFPM1gQKmWXdPBNfigFFQUjRarYk5hFFd9CsCo8HDKcCzYFQYrC8iJDo4YhpbFF2w3Sjx,1773408984,5000,null,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json
2eMTSsX64vspd93GeSffoL84BBo8pvfDwRNpD51446sHFZH3SEpD4rg3wJJa7euTh7qwUkRY33BJgbnwyujWBYv6,1773408975,80000,null,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json
4bW3DTm7hfmaEmpavEHc92XWkunhVJFr76J3rN592wi6H26wpkkbAQWxDKnTxgD8Ky53XCnsKocC223SUmezBVFf,1773408895,80000,List(List(0)),FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json
kGWB1m8K4ZvZYRGncfZMv3A1okHhKMrwacGfUn9FuEnLButrU92rZFj2xqwV48ftYwJQV5qAYwUZrCeecaaSvfU,1773383970,11000,null,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,s3://l1l2bucket/data/raw/phantom/sol_transactions/sol_transactions_20261008T143032_460677Z_a3664059.json


In [0]:
(
    sol_transactions_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.phantom_sol_transactions"
    )
)

saved_transaction_rows = spark.table(
    f"{catalog}.{bronze_schema}.phantom_sol_transactions"
).count()

print(
    "Saved Phantom SOL transaction rows:",
    saved_transaction_rows
)

Saved Phantom SOL transaction rows: 7
